<a href="https://colab.research.google.com/github/adityab-tech/PRISMx/blob/main/notebooks/03_Market_Conditioned_Gating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import yaml
import random
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
!git clone https://github.com/shiyu-coder/Kronos.git

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 371 (delta 66), reused 49 (delta 49), pack-reused 269 (from 1)
Receiving objects: 100% (371/371), 9.30 MiB | 15.57 MiB/s, done.
Resolving deltas: 100% (178/178), done.


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
!pip -q install transformers peft

In [5]:
sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [6]:
from peft import LoraConfig, get_peft_model
from model import Kronos ,KronosTokenizer

In [7]:
PROJECT_ROOT = "/content/drive/MyDrive/PRISM"
TOKENIZER_PATH = f"{PROJECT_ROOT}/models/kronos/tokenizer/best_model"
KRONOS_PATH = f"{PROJECT_ROOT}/models/kronos/basemodel/best_model"
DATASET_PATH = f"{PROJECT_ROOT}/datasets"
RESULTS_PATH = f"{PROJECT_ROOT}/results/del3"
os.makedirs(RESULTS_PATH, exist_ok=True)

In [8]:
#Load Tokenizer
tokenizer = KronosTokenizer.from_pretrained("/content/drive/MyDrive/PRISM/models/kronos/tokenizer/best_model",local_files_only=True)
tokenizer.eval()
print("Tokenizer Loaded")

#Load Fine Tuned Kronos
model = Kronos.from_pretrained( "/content/drive/MyDrive/PRISM/models/kronos/basemodel/best_model",local_files_only=True)
model.eval()
print("Kronos Loaded")

Loading weights from local directory
Tokenizer Loaded
Loading weights from local directory
Kronos Loaded


In [9]:
#Freeze tokenizer
for param in tokenizer.parameters():
    param.requires_grad = False
#Freeze Kronos backbone
for param in model.parameters():
    param.requires_grad = False

In [10]:
lora_config = LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias="none",
    target_modules=["q_proj","k_proj","v_proj","out_proj",])

In [11]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [12]:
model = get_peft_model(model,lora_config)

In [13]:
print(type(tokenizer))
print(type(model))

<class 'model.kronos.KronosTokenizer'>
<class 'peft.peft_model.PeftModel'>


In [14]:
model.print_trainable_parameters()

trainable params: 1,384,448 || all params: 103,695,040 || trainable%: 1.3351


In [15]:
lora_layers = []
for name, _ in model.named_parameters():
    if "lora_" in name:
        lora_layers.append(name)

print("Total LoRA tensors :", len(lora_layers))
print(lora_layers[:10])

Total LoRA tensors : 104
['base_model.model.transformer.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.transformer.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.transformer.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.transformer.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.transformer.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.transformer.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.transformer.0.self_attn.out_proj.lora_A.default.weight', 'base_model.model.transformer.0.self_attn.out_proj.lora_B.default.weight', 'base_model.model.transformer.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.transformer.1.self_attn.q_proj.lora_B.default.weight']


In [16]:
market_df = pd.read_csv("/content/drive/MyDrive/PRISM/data/raw/NIFTY50.csv")
market_df.head()

,Date,Adj Close,Close,High,Low,Open,Volume
0,2019-01-02,10792.500000,10792.500000,10895.349609,10735.049805,10868.849609,309700
1,2019-01-03,10672.250000,10672.250000,10814.049805,10661.250000,10796.799805,286200
2,2019-01-04,10727.349609,10727.349609,10741.049805,10628.650391,10699.700195,296600
3,2019-01-07,10771.799805,10771.799805,10835.950195,10750.150391,10804.849609,269400
4,2019-01-08,10802.150391,10802.150391,10818.450195,10733.250000,10786.250000,277700


In [17]:
FEATURES = ["Close","Volume","Open","High","Low"]
market_features = market_df[FEATURES]

In [18]:
market_tensor = torch.tensor(market_features.values,dtype=torch.float32)
print(market_tensor.shape)

torch.Size([1478, 5])


In [19]:
SEQ_LEN = 36
market_sequences = []
for i in range(len(market_tensor) - SEQ_LEN):
    market_sequences.append( market_tensor[i:i+SEQ_LEN])

market_sequences = torch.stack(market_sequences)
print(market_sequences.shape)

torch.Size([1442, 36, 5])


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [21]:
class StockEncoder(nn.Module):

    def __init__(self, kronos):
        super().__init__()

        self.kronos = kronos

    def forward(
        self,
        s1_ids,
        s2_ids,
        stamp=None,
        padding_mask=None
    ):

        # Kronos already provides transformer context
        _, context = self.kronos.decode_s1(
            s1_ids,
            s2_ids,
            stamp,
            padding_mask
        )

        return context

In [22]:
stock_encoder = StockEncoder(model).to(device)
stock_encoder.eval()

StockEncoder(
  (kronos): PeftModel(
    (base_model): LoraModel(
      (model): Kronos(
        (token_drop): Dropout(p=0.0, inplace=False)
        (embedding): HierarchicalEmbedding(
          (emb_s1): Embedding(1024, 832)
          (emb_s2): Embedding(1024, 832)
          (fusion_proj): Linear(in_features=1664, out_features=832, bias=True)
        )
        (time_emb): TemporalEmbedding(
          (minute_embed): Embedding(60, 832)
          (hour_embed): Embedding(24, 832)
          (weekday_embed): Embedding(7, 832)
          (day_embed): Embedding(32, 832)
          (month_embed): Embedding(13, 832)
        )
        (transformer): ModuleList(
          (0-11): 12 x TransformerBlock(
            (norm1): RMSNorm()
            (self_attn): MultiHeadAttentionWithRoPE(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=832, out_features=832, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace

In [23]:
with torch.no_grad():

    dummy_s1 = torch.randint(
        0,
        model.s1_vocab_size,
        (2,36)
    ).to(device)

    dummy_s2 = torch.randint(
        0,
        model.s2_bits**2,
        (2,36)
    ).to(device)

    context = stock_encoder(
        dummy_s1,
        dummy_s2
    )

print(context.shape)

torch.Size([2, 36, 832])


In [24]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )

        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [25]:
hidden_dim = model.d_model      # Automatically matches Kronos
n_heads = model.n_heads         # Automatically matches Kronos
class MarketEncoder(nn.Module):

    def __init__(
        self,
        input_dim=5,
        hidden_dim=hidden_dim,
        num_layers=2,
        n_heads=n_heads,
        dropout=0.1
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.pos_encoding = PositionalEncoding(hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim*4,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        x = self.encoder(x)

        return x

In [26]:
market_encoder = MarketEncoder().to(device)
print(market_encoder)

MarketEncoder(
  (input_proj): Linear(in_features=5, out_features=832, bias=True)
  (pos_encoding): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=832, out_features=832, bias=True)
        )
        (linear1): Linear(in_features=832, out_features=3328, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3328, out_features=832, bias=True)
        (norm1): LayerNorm((832,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((832,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
)


In [27]:
dummy_market = torch.randn(32,36,5).to(device)
market_embedding = market_encoder(dummy_market)

print(market_embedding.shape)

torch.Size([32, 36, 832])


In [31]:
class MarketCrossAttention(nn.Module):

    """
    Stock tokens attend over Market embeddings.

    Query  -> Stock Representation

    Key    -> Market Representation

    Value  -> Market Representation
    """

    def __init__(

        self,

        d_model,

        n_heads,

        dropout=0.1

    ):

        super().__init__()

        self.cross_attention = nn.MultiheadAttention(

            embed_dim=d_model,

            num_heads=n_heads,

            dropout=dropout,

            batch_first=True

        )

        self.norm1 = nn.LayerNorm(d_model)

        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(

            nn.Linear(d_model, d_model * 4),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(d_model * 4, d_model)

        )

        self.dropout = nn.Dropout(dropout)

    def forward(

        self,

        stock_features,

        market_features,

        market_padding_mask=None

    ):

        residual = stock_features

        attended_features, attention_weights = self.cross_attention(

            query=stock_features,

            key=market_features,

            value=market_features,

            key_padding_mask=market_padding_mask,

            need_weights=True

        )

        x = self.norm1(

            residual +

            self.dropout(attended_features)

        )

        ff = self.ffn(x)

        x = self.norm2(

            x +

            self.dropout(ff)

        )

        return x, attention_weights

In [32]:
cross_attention = MarketCrossAttention(

    d_model=model.d_model,

    n_heads=model.n_heads,

    dropout=0.1

).to(device)

In [33]:
dummy_stock = torch.randn(

    32,

    36,

    model.d_model

).to(device)

dummy_market = torch.randn(

    32,

    36,

    model.d_model

).to(device)

cross_output, attention = cross_attention(

    dummy_stock,

    dummy_market

)

print("Output :", cross_output.shape)

print("Attention :", attention.shape)

Output : torch.Size([32, 36, 832])
Attention : torch.Size([32, 36, 36])


In [38]:
class MasterGate(nn.Module):

    """
    Adaptive Market Conditioning Gate

    Input:
        stock_features  -> (B,T,D)

        market_features -> (B,T,D)

    Output:
        fused_features  -> (B,T,D)
    """

    def __init__(self, d_model, dropout=0.1):

        super().__init__()

        self.stock_proj = nn.Linear(
            d_model,
            d_model
        )

        self.market_proj = nn.Linear(
            d_model,
            d_model
        )

        self.gate = nn.Sequential(

            nn.Linear(
                d_model*2,
                d_model
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                d_model,
                d_model
            ),

            nn.Sigmoid()

        )

        self.norm = nn.LayerNorm(d_model)

    def forward(

        self,

        stock_features,

        market_features

    ):

        stock = self.stock_proj(stock_features)

        market = self.market_proj(market_features)

        gate = self.gate(

            torch.cat(

                [stock, market],

                dim=-1

            )

        )
        delta = stock - market
        fused = (
            stock + gate * delta

        )

        fused = self.norm(fused)

        return fused, gate

In [39]:
gate_head = MasterGate(
    d_model=model.d_model
).to(device)

In [40]:
dummy_stock = torch.randn(
    32,
    36,
    model.d_model
).to(device)

dummy_market = torch.randn(
    32,
    36,
    model.d_model
).to(device)

fused, gate = gate_head(
    dummy_stock,
    dummy_market
)

print("Fused :", fused.shape)

print("Gate :", gate.shape)

print(
    gate.min().item(),
    gate.max().item()
)

Fused : torch.Size([32, 36, 832])
Gate : torch.Size([32, 36, 832])
0.37036654353141785 0.6434425115585327
